In [ ]:
path = "src/llm_agent.py"
src = open(path).read()

old = '''CONFIGS = {
    "config1":  {"prompt": CONFIG_1_PROMPT,  "parser": parse_config1,
                 "description": "Minimal (classification only)"},
    "config2":  {"prompt": CONFIG_2_PROMPT,  "parser": parse_config2,
                 "description": "Verdict + Explanation"},
    "config2b": {"prompt": CONFIG_2B_PROMPT, "parser": parse_config2,
                 "description": "Decision-first classifier (no CoT)"},
    "config2c": {"prompt": CONFIG_2C_PROMPT, "parser": parse_config3,
                 "description": "Post-hoc CoT explanation of a fixed verdict"},
    "config3":  {"prompt": CONFIG_3_PROMPT,  "parser": parse_config3,
                 "description": "Full CoT schema (verdict last)"},
    "config3b": {"prompt": CONFIG_3B_PROMPT, "parser": parse_config3,
                 "description": "CoT with balanced-prior framing"},
}'''

new = '''CONFIGS = {
    "config1":  {"prompt": CONFIG_1_PROMPT,  "parser": parse_config1,
                 "description": "Minimal (classification only)"},
    "config2":  {"prompt": CONFIG_2_PROMPT,  "parser": parse_config2,
                 "description": "Verdict + short explanation (primary classifier)"},
    "config2c": {"prompt": CONFIG_2C_PROMPT, "parser": parse_config3,
                 "description": "Post-hoc CoT explanation of a fixed verdict"},
}'''

if old in src:
    src = src.replace(old, new, 1)
    open(path, "w").write(src)
    print("Removed config2b, config3, config3b from CONFIGS.")
else:
    print("ERROR: CONFIGS block not found. Check manually.")

ERROR: CONFIGS block not found. Check manually.


In [7]:
!python src/llm_agent.py --shap-aligned --configs config1,config2

Traceback (most recent call last):
  File "/content/drive/MyDrive/smishing_project3/src/llm_agent.py", line 48, in <module>
    "config1":  {"prompt": CONFIG_1_PROMPT,  "parser": parse_config1,
                           ^^^^^^^^^^^^^^^
NameError: name 'CONFIG_1_PROMPT' is not defined


In [10]:
!rm -f outputs/llm_agent_results_100*.json
!rm -f outputs/llm_agent_metrics_100*.csv
!rm -f outputs/checkpoint_shap100.json

!python src/llm_agent.py --shap-aligned --configs config1,config2

LLM SMISHING DETECTION  (input=_100  configs=['config1', 'config2'])
  [Ollama] Checking if Ollama is running...
  [Ollama] Ollama not responding (attempt 1/3)
  [Ollama] Restarting Ollama...
  [Ollama] Ollama restarted successfully.
Available models: ['llama3:latest', 'mistral:latest']

Loading test set from outputs/shap_sample.csv ...
Test messages: 100 (ham=50, malicious=50)
NVIDIA L4, 23034 MiB, 0 MiB, 0 %

MODEL 1/2: mistral
  [Warmup] Loading mistral ...
  [Model] mistral is available.
  [Warmup] Attempt 1/3...
  [Warmup] mistral responded in 78.3s.
NVIDIA L4, 23034 MiB, 4496 MiB, 23 %
NAME              ID              SIZE      PROCESSOR    CONTEXT    UNTIL               
mistral:latest    6577803aa9a0    4.5 GB    100% GPU     512        29 minutes from now    

Config 1/2: config1
  Description: Minimal classification prompt
  Processed: 0/100
  [10/100] Acc: 0.800 | 2.01 msg/s | ETA: 0.7min
  [20/100] Acc: 0.850 | 3.23 msg/s | ETA: 0.4min
  [30/100] Acc: 0.867 | 4.06 msg/s | 

In [13]:
import json, os, pandas as pd

# 1. Check the results file
r = json.load(open("outputs/llm_agent_results_100_config1_config2.json"))
print("Total rows:", len(r))
print("Configs:", sorted(set(x["config"] for x in r)))
print("Models:", sorted(set(x["model"] for x in r)))

# 2. Check config2 verdicts are present
c2 = [x for x in r if x["config"] == "config2" and x.get("verdict")]
print(f"\nconfig2 rows with verdict: {len(c2)}")

# 3. Check the metrics CSV
print("\n--- metrics ---")
df = pd.read_csv("outputs/llm_agent_metrics_100_config1_config2.csv")
print(df[["model", "config", "accuracy", "f1", "fpr"]].to_string(index=False))

# 4. Check the checkpoint
ckpt = json.load(open("outputs/checkpoint_shap100.json"))
print("\n--- checkpoint progress ---")
for k, v in ckpt.get("progress", {}).items():
    print(f"  {k}: {v}")

Total rows: 400
Configs: ['config1', 'config2']
Models: ['llama3', 'mistral']

config2 rows with verdict: 200

--- metrics ---
  model  config  accuracy     f1  fpr
mistral config1      0.86 0.8704 0.22
mistral config2      0.91 0.9174 0.18
 llama3 config1      0.84 0.8596 0.30
 llama3 config2      0.87 0.8850 0.26

--- checkpoint progress ---
  mistral_config1: 100
  mistral_config2: 100
  llama3_config1: 100
  llama3_config2: 100


In [1]:
# Install dependencies
!apt-get update -qq && apt-get install -y -qq zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background
import subprocess, time
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(10)  # wait for server to fully start
print("Ollama server started")

# Verify it is running
import requests
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    print("Ollama is reachable:", r.status_code)
except Exception as e:
    print("ERROR:", e)

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> C

In [15]:
path = "src/llm_agent.py"
src = open(path).read()

old = '''def load_config2_verdicts(model):
    """Load verdicts from config2 (the working classifier) for use by config2c."""
    path = f"{OUTPUT_DIR}/llm_agent_results{_INPUT_TAG}.json"
    if not os.path.exists(path):
        print(f"  WARNING: config2 results not found at {path}")
        print(f"  Run this first: python src/llm_agent.py "
              f"{'--shap-aligned ' if _args.shap_aligned else ''}--configs config1,config2")
        return {}
    with open(path) as f:
        data = json.load(f)
    lookup = {}
    for r in data:
        if r['config'] == 'config2' and r['model'] == model and r['verdict']:
            lookup[r['idx']] = r['verdict']
    print(f"  [config2c] Loaded {len(lookup)} verdicts from config2 for {model}")
    return lookup'''

new = '''def load_config2_verdicts(model):
    """Load verdicts from config2 (the working classifier) for use by config2c.

    Searches all llm_agent_results_100*.json files for config2 entries,
    because the filename varies depending on the --configs argument used.
    """
    import glob
    pattern = f"{OUTPUT_DIR}/llm_agent_results{_INPUT_TAG}*.json"
    candidates = sorted(glob.glob(pattern))
    if not candidates:
        print(f"  WARNING: no results files found matching {pattern}")
        return {}
    lookup = {}
    for path in candidates:
        try:
            with open(path) as f:
                data = json.load(f)
        except Exception:
            continue
        for r in data:
            if (r.get('config') == 'config2'
                    and r.get('model') == model
                    and r.get('verdict')):
                lookup[r['idx']] = r['verdict']
    print(f"  [config2c] Loaded {len(lookup)} verdicts from config2 for {model} "
          f"(from {len(candidates)} file(s))")
    return lookup'''

if old in src:
    src = src.replace(old, new, 1)
    open(path, "w").write(src)
    print("Loader updated to search all results files.")
else:
    print("ERROR: function body changed. Manual edit required.")

Loader updated to search all results files.


In [16]:
!python -c "import ast; ast.parse(open('src/llm_agent.py').read()); print('syntax OK')"
!grep -n "glob.glob" src/llm_agent.py

syntax OK
468:    candidates = sorted(glob.glob(pattern))


In [17]:
import os
for f in ["outputs/llm_agent_results_100_config2c.json",
          "outputs/llm_agent_metrics_100_config2c.csv",
          "outputs/llm_agent_cited_tokens_100_config2c.json"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

# Also remove config2c from checkpoint so it does not skip
import json
ckpt_path = "outputs/checkpoint_shap100.json"
if os.path.exists(ckpt_path):
    ckpt = json.load(open(ckpt_path))
    ckpt["results"] = [r for r in ckpt.get("results", []) if r.get("config") != "config2c"]
    ckpt["metrics"] = [m for m in ckpt.get("metrics", []) if m.get("config") != "config2c"]
    ckpt["progress"] = {k: v for k, v in ckpt.get("progress", {}).items()
                        if "config2c" not in k}
    json.dump(ckpt, open(ckpt_path, "w"), indent=2)
    print("Cleaned config2c from checkpoint.")
    print("Remaining progress:", ckpt["progress"])

Removed outputs/llm_agent_results_100_config2c.json
Removed outputs/llm_agent_cited_tokens_100_config2c.json
Cleaned config2c from checkpoint.
Remaining progress: {'mistral_config1': 100, 'mistral_config2': 100, 'llama3_config1': 100, 'llama3_config2': 100}


In [18]:
!python src/llm_agent.py --shap-aligned --configs config2c

LLM SMISHING DETECTION  (input=_100  configs=['config2c'])
  [Ollama] Checking if Ollama is running...
  [Ollama] Ollama not responding (attempt 1/3)
  [Ollama] Restarting Ollama...
  [Ollama] Ollama restarted successfully.
Available models: ['llama3:latest', 'mistral:latest']

Loading test set from outputs/shap_sample.csv ...
Test messages: 100 (ham=50, malicious=50)
NVIDIA L4, 23034 MiB, 0 MiB, 0 %
  [Resume] Loaded checkpoint from 2026-09-23 08:29:20

  [Resume] Progress: {'mistral_config1': 100, 'mistral_config2': 100, 'llama3_config1': 100, 'llama3_config2': 100}

MODEL 1/2: mistral
  [Warmup] Loading mistral ...
  [Model] mistral is available.
  [Warmup] Attempt 1/3...
  [Warmup] mistral responded in 2.9s.
NVIDIA L4, 23034 MiB, 4496 MiB, 58 %
NAME              ID              SIZE      PROCESSOR    CONTEXT    UNTIL               
mistral:latest    6577803aa9a0    4.5 GB    100% GPU     512        29 minutes from now    
  [config2c] Loaded 100 verdicts from config2 for mistral (f

In [2]:
# Pull Mistral-7B (same as your Windows setup)
!ollama pull mistral

# Pull Llama 3-8B (same as your Windows setup)
!ollama pull llama3

# Verify both are available
import requests
r = requests.get("http://localhost:11434/api/tags")
models = [m['name'] for m in r.json().get('models', [])]
print("Available models:", models)



Available models: ['llama3:latest', 'mistral:latest']


In [8]:
from google.colab import files
import shutil, os

# Upload test.csv from your Windows machine
uploaded = files.upload()   # select data/test.csv when prompted

os.makedirs('/content/smishing_project3/data',    exist_ok=True)
os.makedirs('/content/smishing_project3/outputs', exist_ok=True)
os.makedirs('/content/smishing_project3/src',     exist_ok=True)

# Move uploaded file to correct location
shutil.copy('shap_sample.csv', '/content/smishing_project3/data/shap_sample.csv')
print("shap_sample.csv copied to data folder")

Saving shap_sample.csv to shap_sample (4).csv
shap_sample.csv copied to data folder


In [ ]:
# Upload your llm_agent.py
uploaded = files.upload()   # select src/llm_agent.py when prompted
shutil.copy('llm_agent.py', '/content/smishing_project/src/llm_agent.py')
print("llm_agent.py copied to src folder")

Saving llm_agent.py to llm_agent.py
llm_agent.py copied to src folder


In [ ]:
!pip install scikit-learn pandas numpy requests -q

In [3]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [9]:
%%javascript
function KeepAlive() {
    document.querySelector("#connect").click();
    console.log("Keeping Colab alive...");
}
setInterval(KeepAlive, 60000);  // clicks every 60 seconds

<IPython.core.display.Javascript object>

In [4]:
import os
PROJECT = "/content/drive/MyDrive/smishing_project3"
os.chdir(PROJECT)
print("cwd:", os.getcwd())
!ls

cwd: /content/drive/MyDrive/smishing_project3
data  outputs  src


In [12]:
from pathlib import Path
for p in Path("/").rglob("shap_sample.csv"):
    print(p)

/content/drive/MyDrive/smishing_project3/data/shap_sample.csv
/content/drive/MyDrive/smishing_project/outputs/shap_sample.csv


In [6]:
path = "src/llm_agent.py"
src = open(path).read()

old = '''CONFIGS = {
    "config1":  {"prompt": CONFIG_1_PROMPT,  "parser": parse_config1,
                 "description": "Minimal (classification only)"},
    "config2":  {"prompt": CONFIG_2_PROMPT,  "parser": parse_config2,
                 "description": "Verdict + Explanation"},
    "config2b": {"prompt": CONFIG_2B_PROMPT, "parser": parse_config2,
                 "description": "Decision-first classifier (no CoT)"},
    "config2c": {"prompt": CONFIG_2C_PROMPT, "parser": parse_config3,
                 "description": "Post-hoc CoT explanation of a fixed verdict"},
    "config3":  {"prompt": CONFIG_3_PROMPT,  "parser": parse_config3,
                 "description": "Full CoT schema (verdict last)"},
    "config3b": {"prompt": CONFIG_3B_PROMPT, "parser": parse_config3,
                 "description": "CoT with balanced-prior framing"},
}'''

new = '''CONFIGS = {
    "config1":  {"prompt": CONFIG_1_PROMPT,  "parser": parse_config1,
                 "description": "Minimal (classification only)"},
    "config2":  {"prompt": CONFIG_2_PROMPT,  "parser": parse_config2,
                 "description": "Verdict + short explanation (primary classifier)"},
    "config2c": {"prompt": CONFIG_2C_PROMPT, "parser": parse_config3,
                 "description": "Post-hoc CoT explanation of a fixed verdict"},
}'''

if old in src:
    src = src.replace(old, new, 1)
    open(path, "w").write(src)
    print("Removed config2b, config3, config3b from CONFIGS.")
else:
    print("ERROR: CONFIGS block not found. Check manually.")

ERROR: CONFIGS block not found. Check manually.


In [16]:
import os, glob, shutil

# Backup first
os.makedirs("outputs/backup_all_old", exist_ok=True)
for f in glob.glob("outputs/llm_agent_*") + glob.glob("outputs/checkpoint_*"):
    if os.path.isfile(f):
        shutil.copy(f, f"outputs/backup_all_old/{os.path.basename(f)}")
print("Backed up all old results.")

# Remove old results so configs run from scratch
for f in glob.glob("outputs/llm_agent_results_100*.json"):
    os.remove(f); print(f"Removed {f}")
for f in glob.glob("outputs/llm_agent_metrics_100*.csv"):
    os.remove(f); print(f"Removed {f}")
for f in ["outputs/checkpoint_shap100.json"]:
    if os.path.exists(f):
        os.remove(f); print(f"Removed {f}")

# Remove config-specific files that no longer apply
for f in ["outputs/llm_agent_results_100_config2b.json",
          "outputs/llm_agent_results_100_config3.json",
          "outputs/llm_agent_results_100_config3b.json",
          "outputs/llm_agent_metrics_100_config2b.csv",
          "outputs/llm_agent_metrics_100_config3.csv",
          "outputs/llm_agent_metrics_100_config3b.csv"]:
    if os.path.exists(f):
        os.remove(f); print(f"Removed {f}")

print("\nClean.")

Backed up all old results.

Clean.


In [10]:
!grep -n "test_df = pd.read_csv" src/llm_agent.py

612:    test_df = pd.read_csv(TEST_CSV)


In [11]:
!rm -f outputs/llm_agent_results_100_config2b.json
!rm -f outputs/llm_agent_metrics_100_config2b.csv
!rm -f outputs/checkpoint_shap100.json
!rm -rf outputs/ollama_cache_v3/
!rm -rf outputs/ollama_cache/
print("cleaned")

cleaned


In [7]:
!pkill -f ollama || true
import time
time.sleep(3)
!nohup ollama serve > /tmp/ollama.log 2>&1 &
time.sleep(20)
!ollama list

^C
NAME              ID              SIZE      MODIFIED       
llama3:latest     365c0bd3c000    4.7 GB    10 minutes ago    
mistral:latest    6577803aa9a0    4.4 GB    11 minutes ago    


In [19]:
import json
r = json.load(open("outputs/llm_agent_results_100_config2c.json"))
c2c = [x for x in r if x["config"] == "config2c"]
with_tokens = [x for x in c2c if x.get("cited_tokens")]
print(f"config2c rows: {len(c2c)}")
print(f"with cited tokens: {len(with_tokens)}")
print()
for m in ["mistral", "llama3"]:
    sub = [x for x in with_tokens if x["model"] == m]
    print(f"{m}: {len(sub)} rows with tokens")
    if sub:
        ex = sub[0]
        print(f"  msg    : {ex['text'][:70]}")
        print(f"  verdict: {ex['verdict']}")
        print(f"  cited  : {ex['cited_tokens']}")
        print()

config2c rows: 200
with cited tokens: 195

mistral: 100 rows with tokens
  msg    : I'm home. Ard wat time will u reach?
  verdict: LEGITIMATE
  cited  : ['will', 'specific', 'indicates', '2.', 'for', 'currently', 'short', 'reach', 'is', 'act']

llama3: 95 rows with tokens
  msg    : I'm home. Ard wat time will u reach?
  verdict: LEGITIMATE
  cited  : ['important', 'verdict', 'ard', 'for', 'justify', 'way', 'common', 'abbreviation', 'wat', 'most']



In [20]:
path = "src/llm_agent.py"
src = open(path).read()

old = '''def parse_config2c(response):
    """Parser for config2c: extracts STEP 3 CITED WORDS and EXPLANATION."""
    if response is None:
        return {"verdict": None, "confidence": None, "cot_trace": None, "cited_tokens": [],
                "tech_explanation": None, "user_explanation": None, "parse_error": "null_response"}
    result = {"verdict": None, "confidence": None, "cot_trace": response,
              "cited_tokens": [], "tech_explanation": None,
              "user_explanation": None, "parse_error": None}

    # Extract CITED WORDS
    cited_patterns = [
        r'STEP\\s*3[^:]*:\\s*(.*?)(?=EXPLANATION|\\Z)',
        r'CITED WORDS\\s*:?\\s*(.*?)(?=EXPLANATION|\\Z)',
    ]
    for pattern in cited_patterns:
        m = re.search(pattern, response, re.IGNORECASE | re.DOTALL)
        if m:
            text = m.group(1)
            quoted = re.findall(r'["\\u2018\\u2019\\u201c\\u201d]([^"\\']+)["\\u2018\\u2019\\u201c\\u201d]', text)
            bullet = re.findall(r'[•\\-*]\\s*["\\']?([a-zA-Z0-9\\s\\-]+)["\\']?', text)
            all_tokens = []
            for t in quoted + bullet:
                words = [w.strip().lower() for w in t.split() if len(w.strip()) > 1]
                all_tokens.extend(words)
            if not all_tokens:
                all_tokens.extend([w.lower() for w in re.findall(r'\\b[A-Z][a-z]+\\b', text)])
            result["cited_tokens"] = list(set(all_tokens))[:10]
            break

    # Extract EXPLANATION
    m = re.search(r'EXPLANATION\\s*:?\\s*(.*?)$', response, re.IGNORECASE | re.DOTALL)
    if m:
        result["tech_explanation"] = m.group(1).strip()
        result["user_explanation"] = result["tech_explanation"]
    return result'''

new = '''def parse_config2c(response):
    """Parser for config2c: extracts STEP 3 CITED WORDS and EXPLANATION.

    Only tokens that appear verbatim in the source SMS are accepted.
    """
    if response is None:
        return {"verdict": None, "confidence": None, "cot_trace": None, "cited_tokens": [],
                "tech_explanation": None, "user_explanation": None, "parse_error": "null_response"}

    result = {"verdict": None, "confidence": None, "cot_trace": response,
              "cited_tokens": [], "tech_explanation": None,
              "user_explanation": None, "parse_error": None}

    # Extract STEP 3 block (CITED WORDS)
    cited_patterns = [
        r'STEP\\s*3[^:]*:\\s*(.*?)(?=EXPLANATION|\\Z)',
        r'CITED WORDS\\s*:?\\s*(.*?)(?=EXPLANATION|\\Z)',
    ]
    quoted_tokens = []
    for pattern in cited_patterns:
        m = re.search(pattern, response, re.IGNORECASE | re.DOTALL)
        if m:
            text = m.group(1)
            # ONLY take tokens inside quotes — this is the strongest signal
            quoted = re.findall(
                r'["\\u2018\\u2019\\u201c\\u201d]([^"\\']{2,40})["\\u2018\\u2019\\u201c\\u201d]',
                text)
            for t in quoted:
                for w in t.split():
                    w = w.strip().lower()
                    if len(w) > 1:
                        quoted_tokens.append(w)
            break

    # Deduplicate while preserving order
    seen = set()
    cited = []
    for t in quoted_tokens:
        if t not in seen:
            seen.add(t)
            cited.append(t)

    result["cited_tokens"] = cited[:10]

    # Extract EXPLANATION
    m = re.search(r'EXPLANATION\\s*:?\\s*(.*?)$', response, re.IGNORECASE | re.DOTALL)
    if m:
        result["tech_explanation"] = m.group(1).strip()
        result["user_explanation"] = result["tech_explanation"]
    return result'''

if old in src:
    src = src.replace(old, new, 1)
    open(path, "w").write(src)
    print("Parser replaced.")
else:
    print("ERROR: parser not found. Manual edit required.")

Parser replaced.


In [21]:
path = "src/llm_agent.py"
src = open(path).read()

old = """                parsed = config['parser'](response)

                # For config2c, verdict is inherited from config2 (fixed)
                if config_name == 'config2c':
                    parsed['verdict'] = verdict_lookup.get(int(idx))
                    parsed['parse_error'] = None"""

new = """                parsed = config['parser'](response)

                # For config2c, verdict is inherited from config2 (fixed)
                if config_name == 'config2c':
                    parsed['verdict'] = verdict_lookup.get(int(idx))
                    parsed['parse_error'] = None
                    # Keep only cited tokens that appear in the source message
                    msg_words = set(re.findall(r"[a-z0-9']+", msg.lower()))
                    parsed['cited_tokens'] = [
                        t for t in parsed['cited_tokens'] if t in msg_words
                    ]"""

if old in src:
    src = src.replace(old, new, 1)
    open(path, "w").write(src)
    print("Message filter added.")
else:
    print("ERROR: block not found. Manual edit required.")

Message filter added.


In [23]:
!grep -n "msg_words = set" src/llm_agent.py
!grep -n "quoted_tokens" src/llm_agent.py

638:                    msg_words = set(re.findall(r"[a-z0-9']+", msg.lower()))
353:    quoted_tokens = []
366:                        quoted_tokens.append(w)
372:    for t in quoted_tokens:


In [24]:
import os, json
for f in ["outputs/llm_agent_results_100_config2c.json",
          "outputs/llm_agent_metrics_100_config2c.csv",
          "outputs/llm_agent_cited_tokens_100_config2c.json"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

ckpt_path = "outputs/checkpoint_shap100.json"
if os.path.exists(ckpt_path):
    ckpt = json.load(open(ckpt_path))
    ckpt["results"] = [r for r in ckpt.get("results", []) if r.get("config") != "config2c"]
    ckpt["metrics"] = [m for m in ckpt.get("metrics", []) if m.get("config") != "config2c"]
    ckpt["progress"] = {k: v for k, v in ckpt.get("progress", {}).items()
                        if "config2c" not in k}
    json.dump(ckpt, open(ckpt_path, "w"), indent=2)
    print("Checkpoint cleaned.")

Removed outputs/llm_agent_results_100_config2c.json
Removed outputs/llm_agent_metrics_100_config2c.csv
Removed outputs/llm_agent_cited_tokens_100_config2c.json
Checkpoint cleaned.


In [25]:
!python src/llm_agent.py --shap-aligned --configs config2c

LLM SMISHING DETECTION  (input=_100  configs=['config2c'])
  [Ollama] Checking if Ollama is running...
  [Ollama] Ollama is running and responsive.
Available models: ['llama3:latest', 'mistral:latest']

Loading test set from outputs/shap_sample.csv ...
Test messages: 100 (ham=50, malicious=50)
NVIDIA L4, 23034 MiB, 0 MiB, 0 %
  [Resume] Loaded checkpoint from 2026-09-23 08:49:05

  [Resume] Progress: {'mistral_config1': 100, 'mistral_config2': 100, 'llama3_config1': 100, 'llama3_config2': 100}

MODEL 1/2: mistral
  [Warmup] Loading mistral ...
  [Model] mistral is available.
  [Warmup] Attempt 1/3...
  [Warmup] mistral responded in 2.9s.
NVIDIA L4, 23034 MiB, 4496 MiB, 25 %
NAME              ID              SIZE      PROCESSOR    CONTEXT    UNTIL               
mistral:latest    6577803aa9a0    4.5 GB    100% GPU     512        29 minutes from now    
  [config2c] Loaded 100 verdicts from config2 for mistral (from 1 file(s))

Config 1/1: config2c
  Description: Post-hoc CoT explanation

In [26]:
import json
r = json.load(open("outputs/llm_agent_results_100_config2c.json"))
c2c = [x for x in r if x["config"] == "config2c"]
with_tokens = [x for x in c2c if x.get("cited_tokens")]
print(f"config2c rows: {len(c2c)}")
print(f"with cited tokens: {len(with_tokens)}")
print()
for m in ["mistral", "llama3"]:
    sub = [x for x in with_tokens if x["model"] == m]
    print(f"{m}: {len(sub)} rows with tokens")
    for ex in sub[:2]:
        msg_words = set(ex["text"].lower().split())
        in_msg = [t for t in ex["cited_tokens"] if any(t in w for w in msg_words)]
        print(f"  msg  : {ex['text'][:60]}")
        print(f"  cited: {ex['cited_tokens']}")
        print(f"  in-msg ratio: {len(in_msg)}/{len(ex['cited_tokens'])}")
    print()

config2c rows: 200
with cited tokens: 176

mistral: 90 rows with tokens
  msg  : I'm home. Ard wat time will u reach?
  cited: ['ard', 'reach']
  in-msg ratio: 2/2
  msg  : G says you never answer your texts, confirm/deny
  cited: ['you', 'never', 'answer', 'your', 'texts']
  in-msg ratio: 5/5

llama3: 86 rows with tokens
  msg  : G says you never answer your texts, confirm/deny
  cited: ['says', 'you', 'never', 'answer', 'your', 'texts']
  in-msg ratio: 6/6
  msg  : As a valued customer, I am pleased to advise you that follow
  cited: ['as', 'valued', 'customer']
  in-msg ratio: 3/3



In [27]:
#FAITHFULLNESS TEST
import json, re, requests, hashlib, os
import pandas as pd

OLLAMA_URL = "http://localhost:11434/api/generate"
CACHE_DIR = "outputs/faithfulness_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

CLASSIFY_PROMPT = """You are a cybersecurity analyst specialising in SMS phishing detection.

PHISHING: Messages that impersonate trusted organisations to steal credentials, money, or personal data.
LEGITIMATE: Normal everyday SMS messages.

SMS: {message}

Respond in exactly this format:
VERDICT: [PHISHING or LEGITIMATE]"""


def call_ollama(model, prompt, timeout=180):
    key = hashlib.sha256(f"faith||{model}||{prompt}".encode()).hexdigest()
    path = os.path.join(CACHE_DIR, key + ".json")
    if os.path.exists(path):
        return json.load(open(path))["response"]
    try:
        r = requests.post(OLLAMA_URL, json={
            "model": model, "prompt": prompt, "stream": False,
            "options": {"temperature": 0.0, "num_predict": 30},
        }, timeout=timeout)
        r.raise_for_status()
        resp = r.json().get("response", "").strip()
    except Exception:
        resp = ""
    json.dump({"response": resp}, open(path, "w"))
    return resp


def parse_verdict(resp):
    m = re.search(r"VERDICT\s*:?\s*\**\s*(PHISHING|LEGITIMATE)", resp, re.IGNORECASE)
    if m:
        return m.group(1).upper()
    for w in resp.upper().split():
        c = re.sub(r"[^A-Z]", "", w)
        if c in ("PHISHING", "LEGITIMATE"):
            return c
    return None


def mask_tokens(text, tokens):
    out = text
    for t in tokens:
        if not t.strip():
            continue
        out = re.sub(r"\b" + re.escape(t.strip()) + r"\b",
                     "[MASK]", out, flags=re.IGNORECASE)
    return out


r = json.load(open("outputs/llm_agent_results_100_config2c.json"))
c2c = [x for x in r if x["config"] == "config2c" and x.get("cited_tokens")]

print(f"Evaluating {len(c2c)} config2c entries\n")

flips = 0
attempts = 0
records = []

for i, row in enumerate(c2c):
    model = row["model"]
    text = row["text"]
    original_verdict = row["verdict"]
    cited = row["cited_tokens"]

    if not original_verdict or not cited:
        continue

    masked = mask_tokens(text, cited)
    if masked.strip() == text.strip():
        continue

    prompt = CLASSIFY_PROMPT.format(message=masked)
    resp = call_ollama(model, prompt)
    new_verdict = parse_verdict(resp)
    if new_verdict is None:
        continue

    flipped = (new_verdict != original_verdict)
    if flipped:
        flips += 1
    attempts += 1

    records.append({
        "model": model, "idx": row["idx"],
        "original": original_verdict, "after_mask": new_verdict,
        "flipped": flipped, "cited_tokens": cited,
    })

    if (i + 1) % 20 == 0:
        rate = flips / attempts if attempts else 0
        print(f"  [{i+1}/{len(c2c)}] flip rate = {rate:.3f} ({flips}/{attempts})")

print()
print("=" * 60)
print("FAITHFULNESS -- config2c cited tokens")
print("=" * 60)
for m in sorted(set(x["model"] for x in records)):
    sub = [x for x in records if x["model"] == m]
    f = sum(1 for x in sub if x["flipped"])
    n = len(sub)
    print(f"{m}: {f}/{n} flipped = {f/n if n else 0:.3f}")

pd.DataFrame(records).to_csv("outputs/faithfulness_config2c.csv", index=False)
print("Saved: outputs/faithfulness_config2c.csv")

Evaluating 176 config2c entries

  [20/176] flip rate = 0.400 (8/20)
  [40/176] flip rate = 0.350 (14/40)
  [60/176] flip rate = 0.333 (20/60)
  [80/176] flip rate = 0.362 (29/80)
  [100/176] flip rate = 0.333 (33/99)
  [120/176] flip rate = 0.319 (38/119)
  [140/176] flip rate = 0.295 (41/139)
  [160/176] flip rate = 0.302 (48/159)

FAITHFULNESS -- config2c cited tokens
llama3: 21/85 flipped = 0.247
mistral: 32/89 flipped = 0.360
Saved: outputs/faithfulness_config2c.csv


In [28]:
import shutil, os

src = "/content/drive/MyDrive/smishing_project/models/distilbert_final"
dst = "/content/drive/MyDrive/smishing_project3/models/distilbert_final"

# Create target dir if it does not exist
os.makedirs(os.path.dirname(dst), exist_ok=True)

# Copy (do not move yet — safer to keep the original until verified)
if os.path.exists(src):
    if os.path.exists(dst):
        print("Destination already exists. Skipping.")
    else:
        shutil.copytree(src, dst)
        print(f"Copied to {dst}")
else:
    print(f"Source not found: {src}")

Copied to /content/drive/MyDrive/smishing_project3/models/distilbert_final


In [29]:
import shutil, os
for f in ["distilbert_shap_values.pkl","distilbert_results.csv"]:
    src = f"/content/drive/MyDrive/smishing_project/outputs/{f}"
    dst = f"/content/drive/MyDrive/smishing_project3/outputs/{f}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f"Copied {f}")
    elif os.path.exists(dst):
        print(f"{f} already present in project3")
    else:
        print(f"{f}: source missing")

Copied distilbert_shap_values.pkl
Copied distilbert_results.csv


In [31]:
!python src/eraser_distilbert_flips.py

Loading DistilBERT ...
Loading weights: 100% 104/104 [00:00<00:00, 3715.60it/s]
Model loaded.

Loading SHAP values ...
Sample: 100 messages (ham=50, malicious=50)

Computing flip rates ...

  [20/100] flip rate = 0.000 (0/18)
  [40/100] flip rate = 0.000 (0/37)
  [60/100] flip rate = 0.000 (0/57)
  [80/100] flip rate = 0.000 (0/77)
  [100/100] flip rate = 0.010 (1/96)

ERASER FLIP RATE -- DistilBERT + SHAP top-5
Overall flip rate : 0.010  (1/96)
Phishing flip rate: 0.000
Ham flip rate     : 0.020

Comparison:
  SHAP + DistilBERT       : 0.010
  LLM cited + Mistral     : 0.360
  LLM cited + Llama3      : 0.247

Saved: outputs/eraser_distilbert_flips.csv


In [34]:
!cd /content/drive/MyDrive && zip -r smishing_project.zip smishing_project -x "*/ollama_cache/*" -x "*/faithfulness_cache/*" -x "*/backup_*/*"

updating: smishing_project/ (stored 0%)
updating: smishing_project/data/ (stored 0%)
updating: smishing_project/data/Dataset_5971.csv (deflated 60%)
updating: smishing_project/data/train.csv (deflated 59%)
updating: smishing_project/data/val.csv (deflated 57%)
updating: smishing_project/data/test.csv (deflated 57%)
updating: smishing_project/models/ (stored 0%)
updating: smishing_project/models/distilbert_final/ (stored 0%)
updating: smishing_project/models/distilbert_final/config.json (deflated 49%)
updating: smishing_project/models/distilbert_final/model.safetensors (deflated 8%)
updating: smishing_project/models/distilbert_final/training_args.bin (deflated 53%)
updating: smishing_project/models/distilbert_final/tokenizer_config.json (deflated 43%)
updating: smishing_project/models/distilbert_final/tokenizer.json (deflated 71%)
updating: smishing_project/models/lr_model.pkl (deflated 16%)
updating: smishing_project/models/lr_vectorizer.pkl (deflated 74%)
updating: smishing_project/mo

In [37]:
from google.colab import files
files.download("/content/drive/MyDrive/smishing_project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [39]:
# collect_all_results.py -- search all smishing* folders in MyDrive
# and print consolidated results for the report.

import os, glob
import pandas as pd

ROOT = "/content/drive/MyDrive"

# ── Find all smishing* folders ────────────────────────────────────────────────
smishing_dirs = sorted([
    os.path.join(ROOT, d)
    for d in os.listdir(ROOT)
    if d.startswith("smishing") and os.path.isdir(os.path.join(ROOT, d))
])

print("=" * 70)
print("SMISHING FOLDERS FOUND")
print("=" * 70)
for d in smishing_dirs:
    print(f"  {d}")
print()


# ── Helper: find a file inside any smishing folder ────────────────────────────
def find_file(relative_path):
    """Return the full path to relative_path inside any smishing folder.
    Returns the first match. Prints all matches if more than one exists."""
    matches = []
    for d in smishing_dirs:
        candidate = os.path.join(d, relative_path)
        if os.path.exists(candidate):
            matches.append(candidate)
    if not matches:
        return None
    if len(matches) > 1:
        print(f"  (multiple matches for {relative_path}: using {matches[0]})")
    return matches[0]


# ── Files we need ─────────────────────────────────────────────────────────────
CLASSIFICATION_FILES = [
    "outputs/distilbert_results.csv",
    "outputs/lr_results.csv",
    "outputs/rf_results.csv",
    "outputs/xgb_results.csv",
    "outputs/llm_agent_metrics_100_config1_config2.csv",
    "outputs/llm_agent_metrics_100_config2c.csv",
    "outputs/eraser_distilbert_summary.csv",
]

FLIP_FILES = [
    "outputs/eraser_distilbert_flips.csv",
    "outputs/faithfulness_config2c.csv",
]


# ── Section 1: classification and SHAP probability tables ────────────────────
print("=" * 70)
print("CLASSIFICATION AND SHAP TABLES")
print("=" * 70)

for rel in CLASSIFICATION_FILES:
    path = find_file(rel)
    print(f"\n===== {rel} =====")
    if path:
        print(f"  source: {path}")
        try:
            df = pd.read_csv(path)
            print(df.to_string(index=False))
        except Exception as e:
            print(f"  ERROR: {e}")
    else:
        print("  NOT FOUND in any smishing folder")


# ── Section 2: flip rates ─────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("FLIP RATES (faithfulness)")
print("=" * 70)

# SHAP + DistilBERT
print("\n--- SHAP + DistilBERT ---")
path = find_file("outputs/eraser_distilbert_flips.csv")
if path:
    print(f"  source: {path}")
    d1 = pd.read_csv(path)
    print(f"  Overall flip rate : {d1['flipped'].mean():.4f}  "
          f"({d1['flipped'].sum()}/{len(d1)})")
    if "label" in d1.columns:
        for lab, name in [(1, "phishing"), (0, "ham")]:
            sub = d1[d1["label"] == lab]
            if len(sub):
                print(f"  {name:8s} flip rate: {sub['flipped'].mean():.4f}  "
                      f"({sub['flipped'].sum()}/{len(sub)})")
else:
    print("  NOT FOUND")

# LLM cited tokens
print("\n--- LLM cited tokens (config2c) ---")
path = find_file("outputs/faithfulness_config2c.csv")
if path:
    print(f"  source: {path}")
    d2 = pd.read_csv(path)
    for m in sorted(d2["model"].unique()):
        sub = d2[d2["model"] == m]
        print(f"  {m:8s}: {sub['flipped'].mean():.4f}  "
              f"({sub['flipped'].sum()}/{len(sub)})")
else:
    print("  NOT FOUND")


# ── Section 3: summary comparison table ──────────────────────────────────────
print("\n" + "=" * 70)
print("FAITHFULNESS COMPARISON")
print("=" * 70)
print(f"\n{'Method':<28} {'Model':<12} {'Flip rate':>10}")
print("-" * 52)

if path:
    d1 = pd.read_csv(find_file("outputs/eraser_distilbert_flips.csv"))
    print(f"{'SHAP top-5':<28} {'DistilBERT':<12} {d1['flipped'].mean():>10.4f}")

path = find_file("outputs/faithfulness_config2c.csv")
if path:
    d2 = pd.read_csv(path)
    for m in sorted(d2["model"].unique()):
        sub = d2[d2["model"] == m]
        label = f"LLM cited tokens"
        print(f"{label:<28} {m:<12} {sub['flipped'].mean():>10.4f}")


# ── Section 4: full inventory of CSVs across all smishing folders ────────────
print("\n" + "=" * 70)
print("ALL CSV FILES ACROSS SMISHING FOLDERS")
print("=" * 70)
for d in smishing_dirs:
    csvs = sorted(glob.glob(os.path.join(d, "outputs", "*.csv")))
    if csvs:
        print(f"\n{d}:")
        for c in csvs:
            size = os.path.getsize(c)
            print(f"  {os.path.basename(c):<55} {size:>8} bytes")


print("\nDone.")

SMISHING FOLDERS FOUND
  /content/drive/MyDrive/smishing_outputs
  /content/drive/MyDrive/smishing_project
  /content/drive/MyDrive/smishing_project2
  /content/drive/MyDrive/smishing_project3

CLASSIFICATION AND SHAP TABLES
  (multiple matches for outputs/distilbert_results.csv: using /content/drive/MyDrive/smishing_project/outputs/distilbert_results.csv)

===== outputs/distilbert_results.csv =====
  source: /content/drive/MyDrive/smishing_project/outputs/distilbert_results.csv
     model  accuracy  precision   recall       f1      fpr  tn  fp  fn  tp
DistilBERT  0.994286   0.993197 0.973333 0.983165 0.001379 724   1   4 146

===== outputs/lr_results.csv =====
  source: /content/drive/MyDrive/smishing_project/outputs/lr_results.csv
              model  accuracy  precision  recall       f1      fpr  tn  fp  fn  tp
Logistic Regression     0.992   0.993103    0.96 0.976271 0.001379 724   1   6 144

===== outputs/rf_results.csv =====
  source: /content/drive/MyDrive/smishing_project/outpu